In [1]:
from pathlib import Path
import sys

from IPython.display import display
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_data import DEFAULT_SPLIT_CONFIG, prepare_split

print("Project:", PROJECT_ROOT)
print("Data module:", SRC_DIR / "ml_project/epic_data.py")
print("Config:", DEFAULT_SPLIT_CONFIG)


Project: D:\Projects\EPIC\EPIC
Data module: D:\Projects\EPIC\EPIC\src\ml_project\epic_data.py
Config: SplitConfig(assembly='jaNemVect1.1', split_version='contig_holdout_v1', seed=42, validation_contig_fraction=0.25)


In [2]:
import torch
import torch.nn as nn
import torch_directml

from ml_project.epic_cnn import (
    CNNBackbone,
    CNNBaseline,
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    ResidualBlock,
    create_cnn_baseline,
    dml_bce_with_logits,
    dml_weighted_profile_bce_with_logits,
    forward_both_strands,
    prepare_oriented_cnn_batch,
)

In [3]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))

DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [4]:
x = torch.randn(32, 3, 128, 128).to(device)

conv = nn.Conv2d(
    in_channels=3,
    out_channels=16,
    kernel_size=3,
    padding=1
).to(device)

y = conv(x)
loss = y.square().mean()
loss.backward()

print("Вход:", x.device)
print("Веса:", conv.weight.device)
print("Результат:", y.device)
print("Градиенты:", conv.weight.grad.device)
print("Loss:", loss.item())

Вход: privateuseone:0
Веса: privateuseone:0
Результат: privateuseone:0
Градиенты: privateuseone:0
Loss: 0.35204988718032837


In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/ml_project").is_dir()
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ml_project.epic_data import load_split

split = load_split(PROJECT_ROOT)

positive_rows = split.positive_targets("train")
sample_row = positive_rows.iloc[[0]]

display(sample_row)

,contig,coordinate_0based,strand,r1_count,r2_count,count,target,template_index,source_split,split
0,NC_064035.1,886447,+,0,2,2,1,17031126,train,train


In [6]:
import numpy as np

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.sequence_context import (
    CONTEXT_BASES,
    extract_oriented_context,
)

sequences, fasta_alphabet = load_baseline_sequences(split)

offsets = np.arange(-10, 11)

context_codes = extract_oriented_context(
    sequences,
    sample_row,
    offsets,
)

dna = "".join(CONTEXT_BASES[code] for code in context_codes[0])

print("DNA:", f"{dna[:10]}[{dna[10]}]{dna[11:]}")
print("Коды:", context_codes)
print("Форма:", context_codes.shape)


DNA: TTTCAAAAGT[A]GTAAGGACTT
Коды: [[3 3 3 1 0 0 0 0 2 3 0 2 3 0 0 2 2 0 1 3 3]]
Форма: (1, 21)


In [7]:
import torch.nn.functional as F

codes = torch.from_numpy(context_codes).long()

x = F.one_hot(
    codes,
    num_classes=5,
).permute(0, 2, 1).float()

print("Форма входа:", x.shape)
print("Центральный нуклеотид:", x[0, :, 10])
print("Суммы по позициям:", x.sum(dim=1))

Форма входа: torch.Size([1, 5, 21])
Центральный нуклеотид: tensor([1., 0., 0., 0., 0.])
Суммы по позициям: tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1.]])


In [8]:
stem = nn.Conv1d(
    in_channels=5,
    out_channels=32,
    kernel_size=1,
).to(device)

x_gpu = x.to(device)
features = stem(x_gpu)

print("Вход:", x_gpu.shape)
print("После stem:", features.shape)
print("Веса:", stem.weight.shape)

Вход: torch.Size([1, 5, 21])
После stem: torch.Size([1, 32, 21])
Веса: torch.Size([32, 5, 1])


In [9]:
block1 = ResidualBlock(
    channels=32,
    kernel_size=7,
    dilation=1,
).to(device)

features_after_block1 = block1(features)

print("До блока:", features.shape)
print("После блока:", features_after_block1.shape)
print("Устройство:", features_after_block1.device)

До блока: torch.Size([1, 32, 21])
После блока: torch.Size([1, 32, 21])
Устройство: privateuseone:0


In [10]:
block2 = ResidualBlock(
    channels=32,
    kernel_size=3,
    dilation=2,
).to(device)

features_after_block2 = block2(features_after_block1)

print("До блока 2:", features_after_block1.shape)
print("После блока 2:", features_after_block2.shape)
print("Устройство:", features_after_block2.device)

До блока 2: torch.Size([1, 32, 21])
После блока 2: torch.Size([1, 32, 21])
Устройство: privateuseone:0


In [11]:
block3 = ResidualBlock(
    channels=32,
    kernel_size=3,
    dilation=4,
).to(device)

features_after_block3 = block3(features_after_block2)

print("До блока 3:", features_after_block2.shape)
print("После блока 3:", features_after_block3.shape)
print("Устройство:", features_after_block3.device)

До блока 3: torch.Size([1, 32, 21])
После блока 3: torch.Size([1, 32, 21])
Устройство: privateuseone:0


In [12]:
block4 = ResidualBlock(
    channels=32,
    kernel_size=3,
    dilation=8,
).to(device)

features_after_block4 = block4(features_after_block3)

print("До блока 4:", features_after_block3.shape)
print("После блока 4:", features_after_block4.shape)
print("Устройство:", features_after_block4.device)

До блока 4: torch.Size([1, 32, 21])
После блока 4: torch.Size([1, 32, 21])
Устройство: privateuseone:0


In [13]:
block5 = ResidualBlock(
    channels=32,
    kernel_size=3,
    dilation=16,
).to(device)

features_after_block5 = block5(features_after_block4)

print("До блока 5:", features_after_block4.shape)
print("После блока 5:", features_after_block5.shape)
print("Устройство:", features_after_block5.device)

До блока 5: torch.Size([1, 32, 21])
После блока 5: torch.Size([1, 32, 21])
Устройство: privateuseone:0


In [14]:
block6 = ResidualBlock(
    channels=32,
    kernel_size=3,
    dilation=32,
).to(device)

features_after_block6 = block6(features_after_block5)

print("До блока 6:", features_after_block5.shape)
print("После блока 6:", features_after_block6.shape)
print("Устройство:", features_after_block6.device)

До блока 6: torch.Size([1, 32, 21])
После блока 6: torch.Size([1, 32, 21])
Устройство: privateuseone:0


In [15]:
backbone = CNNBackbone(
    stem=stem,
    blocks=[
        block1,
        block2,
        block3,
        block4,
        block5,
        block6,
    ],
).to(device)

In [16]:
backbone_features, activations = backbone(
    x_gpu,
    return_activations=True,
)

for name, activation in activations.items():
    print(name, activation.shape)

print("\nПервые параметры:")

for name, parameter in list(backbone.named_parameters())[:8]:
    print(name, parameter.shape)

stem torch.Size([1, 32, 21])
block1 torch.Size([1, 32, 21])
block2 torch.Size([1, 32, 21])
block3 torch.Size([1, 32, 21])
block4 torch.Size([1, 32, 21])
block5 torch.Size([1, 32, 21])
block6 torch.Size([1, 32, 21])

Первые параметры:
stem.weight torch.Size([32, 5, 1])
stem.bias torch.Size([32])
blocks.0.bn1.weight torch.Size([32])
blocks.0.bn1.bias torch.Size([32])
blocks.0.conv1.weight torch.Size([32, 32, 7])
blocks.0.conv1.bias torch.Size([32])
blocks.0.bn2.weight torch.Size([32])
blocks.0.bn2.bias torch.Size([32])


In [17]:
profile_head = nn.Conv1d(
    in_channels=32,
    out_channels=1,
    kernel_size=1,
).to(device)

profile_logits = profile_head(backbone_features)

print("Признаки:", backbone_features.shape)
print("Выход:", profile_logits.shape)
print("Веса головы:", profile_head.weight.shape)
print("Устройство:", profile_logits.device)

Признаки: torch.Size([1, 32, 21])
Выход: torch.Size([1, 1, 21])
Веса головы: torch.Size([1, 32, 1])
Устройство: privateuseone:0


In [18]:
model = CNNBaseline(
    backbone=backbone,
    profile_head=profile_head,
).to(device)

In [19]:
logits, activations = model(
    x_gpu,
    return_activations=True,
)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Выход модели:", logits.shape)
print("Устройство:", logits.device)
print("Receptive field: 261")
print("Обучаемых параметров:", parameter_count)

Выход модели: torch.Size([1, 1, 21])
Устройство: privateuseone:0
Receptive field: 261
Обучаемых параметров: 46433


In [20]:
offsets = np.arange(-130, 131)

context_codes_261 = extract_oriented_context(
    sequences,
    sample_row,
    offsets,
)

codes_261 = torch.from_numpy(context_codes_261).long()

x_261 = F.one_hot(
    codes_261,
    num_classes=5,
).permute(0, 2, 1).float().to(device)

target_center = torch.tensor(
    sample_row["target"].to_numpy(),
    dtype=torch.float32,
).reshape(-1, 1).to(device)

print("Вход:", x_261.shape)
print("Центральный target:", target_center)
print("Центральный индекс:", len(offsets) // 2)

Вход: torch.Size([1, 5, 261])
Центральный target: tensor([[1.]], device='privateuseone:0')
Центральный индекс: 130


In [21]:
FLANK = 130
offsets = np.arange(-FLANK, FLANK + 1)

context_codes_261 = extract_oriented_context(
    sequences,
    sample_row,
    offsets,
)

codes_batch = torch.from_numpy(
    context_codes_261
).long()

x_batch = F.one_hot(
    codes_batch,
    num_classes=5,
).permute(0, 2, 1).float()

target_batch = torch.from_numpy(
    sample_row["target"].to_numpy(dtype=np.float32)
).reshape(-1, 1)

count_batch = torch.from_numpy(
    sample_row["count"].to_numpy(dtype=np.float32)
).reshape(-1, 1)

print("Коды:", codes_batch.shape)
print("CNN-вход:", x_batch.shape)
print("Target:", target_batch)
print("Count:", count_batch)
print("Центр one-hot:", x_batch[0, :, FLANK])

assert x_batch.shape == (len(sample_row), 5, 261)
assert torch.all(x_batch.sum(dim=1) == 1)
assert torch.equal(target_batch, (count_batch > 0).float())

Коды: torch.Size([1, 261])
CNN-вход: torch.Size([1, 5, 261])
Target: tensor([[1.]])
Count: tensor([[2.]])
Центр one-hot: tensor([1., 0., 0., 0., 0.])


In [22]:
import pandas as pd

position_batch = next(
    split.iter_batches(
        "train",
        batch_size=10_000,
    )
)

negative_row = (
    position_batch
    .loc[position_batch["target"] == 0]
    .iloc[[0]]
)

columns = [
    "template_index",
    "contig",
    "coordinate_0based",
    "strand",
    "count",
    "target",
]

training_rows = pd.concat(
    [
        sample_row[columns],
        negative_row[columns],
    ],
    ignore_index=True,
)

display(training_rows)

,template_index,contig,coordinate_0based,strand,count,target
0,17031126,NC_064035.1,886447,+,2,1
1,16974557,NC_064035.1,2268,+,0,0


In [23]:
context_codes_batch = extract_oriented_context(
    sequences,
    training_rows,
    offsets,
)

x_batch = F.one_hot(
    torch.from_numpy(context_codes_batch).long(),
    num_classes=5,
).permute(0, 2, 1).float()

target_batch = torch.from_numpy(
    training_rows["target"].to_numpy(dtype=np.float32)
).reshape(-1, 1)

count_batch = torch.from_numpy(
    training_rows["count"].to_numpy(dtype=np.float32)
).reshape(-1, 1)

print("Вход:", x_batch.shape)
print("Target:", target_batch.squeeze(1))
print("Count:", count_batch.squeeze(1))

Вход: torch.Size([2, 5, 261])
Target: tensor([1., 0.])
Count: tensor([2., 0.])


### DirectML-совместимое обучение

`dml_bce_with_logits` и `DirectMLAdam` импортированы из `ml_project.epic_cnn`.

In [24]:
import warnings

model.train()

x_gpu = x_batch.to(device)
target_gpu = target_batch.to(device)

optimizer = DirectMLAdam(
    model.parameters(),
    lr=1e-3,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.0,
)

head_weights_before = (
    model.profile_head.weight
    .detach()
    .clone()
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    optimizer.zero_grad(set_to_none=True)

    all_logits = model(x_gpu)
    center_logits = all_logits[:, :, FLANK]

    loss = dml_bce_with_logits(
        center_logits,
        target_gpu,
    )

    loss.backward()
    optimizer.step()

fallback_warnings = [
    str(warning.message)
    for warning in caught
    if "fall back to run on the CPU" in str(warning.message)
]

optimizer_state_devices = sorted({
    str(value.device)
    for state in optimizer.state.values()
    for value in state.values()
    if torch.is_tensor(value)
})

weight_change = (
    model.profile_head.weight.detach()
    - head_weights_before
).abs().max().item()

print("Модель:", next(model.parameters()).device)
print("Вход:", x_gpu.device)
print("Logits:", all_logits.device)
print("Loss:", loss.device, loss.item())
print("Градиент:", model.backbone.stem.weight.grad.device)
print("Состояния Adam:", optimizer_state_devices)
print("Изменение весов:", weight_change)
print("CPU fallback:", fallback_warnings)

assert optimizer_state_devices == ["privateuseone:0"]
assert weight_change > 0
assert len(fallback_warnings) == 0

Модель: privateuseone:0
Вход: privateuseone:0
Logits: privateuseone:0
Loss: privateuseone:0 0.8790180683135986
Градиент: privateuseone:0
Состояния Adam: ['privateuseone:0']
Изменение весов: 0.0010000020265579224
CPU fallback: []


In [25]:
DIAGNOSTIC_PER_CLASS = 32
DIAGNOSTIC_SEED = 42

columns = [
    "template_index",
    "contig",
    "coordinate_0based",
    "strand",
    "count",
    "target",
]

diagnostic_positive_rows = (
    positive_rows[columns]
    .sample(
        n=DIAGNOSTIC_PER_CLASS,
        random_state=DIAGNOSTIC_SEED,
    )
)

position_batch = next(
    split.iter_batches(
        "train",
        batch_size=100_000,
    )
)

diagnostic_negative_rows = (
    position_batch
    .loc[position_batch["target"] == 0, columns]
    .sample(
        n=DIAGNOSTIC_PER_CLASS,
        random_state=DIAGNOSTIC_SEED,
    )
)

diagnostic_rows = (
    pd.concat(
        [
            diagnostic_positive_rows,
            diagnostic_negative_rows,
        ],
        ignore_index=True,
    )
    .sample(
        frac=1,
        random_state=DIAGNOSTIC_SEED,
    )
    .reset_index(drop=True)
)

display(
    diagnostic_rows.groupby("target")
    .agg(
        positions=("target", "size"),
        count_sum=("count", "sum"),
    )
)

,positions,count_sum
target,,
0,32,0
1,32,76


In [26]:
diagnostic_x, diagnostic_target, diagnostic_count = (
    prepare_oriented_cnn_batch(
        sequences,
        diagnostic_rows,
        flank=FLANK,
    )
)

print("Вход:", diagnostic_x.shape)
print("Target:", diagnostic_target.shape)
print("Count:", diagnostic_count.shape)

print(
    "Положительных:",
    int(diagnostic_target.sum().item()),
)

print(
    "Все позиции one-hot:",
    bool(torch.all(diagnostic_x.sum(dim=1) == 1)),
)

assert diagnostic_x.shape == (64, 5, 261)
assert diagnostic_target.shape == (64, 1)
assert diagnostic_count.shape == (64, 1)
assert diagnostic_target.sum().item() == 32
assert torch.all(diagnostic_x.sum(dim=1) == 1)

Вход: torch.Size([64, 5, 261])
Target: torch.Size([64, 1])
Count: torch.Size([64, 1])
Положительных: 32
Все позиции one-hot: True


In [27]:
module_model = create_cnn_baseline().to(device)
module_model.eval()

with torch.no_grad():
    module_logits = module_model(
        diagnostic_x[:2].to(device)
    )

parameter_count = sum(
    parameter.numel()
    for parameter in module_model.parameters()
    if parameter.requires_grad
)

print("Выход:", module_logits.shape)
print("Устройство:", module_logits.device)
print("Параметров:", parameter_count)

Выход: torch.Size([2, 1, 261])
Устройство: privateuseone:0
Параметров: 46433


In [28]:
train_positive = split.positive_targets("train")
validation_positive = split.positive_targets("validation")

display(split.summary)

print("Train positives:", len(train_positive))
print("Validation positives:", len(validation_positive))

train_contigs = set(split.intervals("train")["contig"])
validation_contigs = set(split.intervals("validation")["contig"])

print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))
print("Общие contigs:", train_contigs & validation_contigs)

assert train_contigs.isdisjoint(validation_contigs)
assert len(train_positive) > 0
assert len(validation_positive) > 0

,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


Train positives: 170443
Validation positives: 56205
Train contigs: 9
Validation contigs: 3
Общие contigs: set()


In [29]:
profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)

train_batch = train_generator.sample_batch(
    batch_size=4,
)

print(
    "Число train-блоков:",
    len(train_generator.tile_index),
)

print(
    "Обычный вход:",
    train_batch.x.shape,
)

print(
    "Обе ориентации:",
    train_batch.x_both_strands.shape,
)

print(
    "Target:",
    train_batch.target.shape,
)

print(
    "Count:",
    train_batch.count.shape,
)

print(
    "Mask:",
    train_batch.mask.shape,
)

print(
    "Разрешённых position-strand:",
    int(train_batch.mask.sum()),
)

print(
    "Положительных:",
    int(train_batch.target.sum()),
)

print(
    "Sampling branches:",
    train_batch.sampling_branch,
)

print(
    "Сумма loss-весов:",
    train_batch.loss_weight.sum().item(),
)

display(
    train_batch.tiles[
        [
            "tile_id",
            "contig",
            "target_start",
            "allowed_position_strand",
            "positive_positions",
            "sampling_branch",
            "q_mixture",
        ]
    ]
)

Число train-блоков: 130525
Обычный вход: torch.Size([4, 5, 1284])
Обе ориентации: torch.Size([8, 5, 1284])
Target: torch.Size([4, 2, 1024])
Count: torch.Size([4, 2, 1024])
Mask: torch.Size([4, 2, 1024])
Разрешённых position-strand: 6860
Положительных: 33
Sampling branches: ('negative', 'positive', 'positive', 'negative')
Сумма loss-весов: 0.9999999403953552


,tile_id,contig,target_start,allowed_position_strand,positive_positions,sampling_branch,q_mixture
0,111696,NC_064047.1,5334016,996,0,negative,0.000002
1,94385,NC_064043.1,15195136,2048,17,positive,0.000054
2,53353,NC_064039.1,2344960,1768,16,positive,0.000051
3,90192,NC_064043.1,10460160,2048,0,negative,0.000004


In [30]:
import warnings

profile_model = create_cnn_baseline().to(
    device
)

optimizer = DirectMLAdam(
    profile_model.parameters(),
    lr=1e-3,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.0,
)

profile_model.train()

# Уже объединённые на CPU прямая и RC-ориентации.
x_both_gpu = (
    train_batch
    .x_both_strands
    .to(device)
)

# Все выравнивания targets выполнены на CPU.
plus_target_gpu = (
    train_batch
    .plus_target
    .to(device)
)

minus_target_reverse_gpu = (
    train_batch
    .minus_target_reverse
    .to(device)
)

plus_weight_gpu = (
    train_batch
    .plus_loss_weight
    .to(device)
)

minus_weight_reverse_gpu = (
    train_batch
    .minus_loss_weight_reverse
    .to(device)
)

head_before = (
    profile_model
    .profile_head
    .weight
    .detach()
    .clone()
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    (
        plus_logits,
        minus_reverse_logits,
    ) = forward_both_strands(
        profile_model,
        x_both_gpu,
        batch_size=len(train_batch.tiles),
        context_flank=(
            profile_config.context_flank
        ),
        target_length=(
            profile_config.target_length
        ),
    )

    loss = (
        dml_weighted_profile_bce_with_logits(
            plus_logits,
            minus_reverse_logits,
            plus_target_gpu,
            minus_target_reverse_gpu,
            plus_weight_gpu,
            minus_weight_reverse_gpu,
        )
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    loss.backward()
    optimizer.step()

fallback_warnings = [
    str(warning.message)
    for warning in caught
    if "fall back" in str(warning.message).lower()
]

weight_change = (
    profile_model
    .profile_head
    .weight
    .detach()
    .sub(head_before)
    .abs()
    .max()
    .item()
)

adam_devices = sorted(
    {
        str(value.device)
        for state in optimizer.state.values()
        for value in state.values()
        if torch.is_tensor(value)
    }
)

print(
    "Общий вход:",
    x_both_gpu.shape,
    x_both_gpu.device,
)

print(
    "Plus logits:",
    plus_logits.shape,
    plus_logits.device,
)

print(
    "Minus reverse logits:",
    minus_reverse_logits.shape,
    minus_reverse_logits.device,
)

print(
    "Plus target:",
    plus_target_gpu.shape,
    plus_target_gpu.device,
)

print(
    "Minus reverse target:",
    minus_target_reverse_gpu.shape,
    minus_target_reverse_gpu.device,
)

print(
    "Loss:",
    loss.item(),
    loss.device,
)

print(
    "Градиент stem:",
    profile_model
    .backbone
    .stem
    .weight
    .grad
    .device,
)

print(
    "Состояния Adam:",
    adam_devices,
)

print(
    "Изменение весов:",
    weight_change,
)

print(
    "CPU fallback:",
    fallback_warnings,
)

assert plus_logits.shape == (
    4,
    1,
    1024,
)

assert minus_reverse_logits.shape == (
    4,
    1,
    1024,
)

assert torch.isfinite(loss)
assert weight_change > 0
assert not fallback_warnings

Общий вход: torch.Size([8, 5, 1284]) privateuseone:0
Plus logits: torch.Size([4, 1, 1024]) privateuseone:0
Minus reverse logits: torch.Size([4, 1, 1024]) privateuseone:0
Plus target: torch.Size([4, 1, 1024]) privateuseone:0
Minus reverse target: torch.Size([4, 1, 1024]) privateuseone:0
Loss: 0.773625910282135 privateuseone:0
Градиент stem: privateuseone:0
Состояния Adam: ['privateuseone:0']
Изменение весов: 0.0010000020265579224
CPU fallback: []


In [31]:
import time
import warnings

TRAIN_STEPS = 200
TRAIN_BATCH_SIZE = 4
LOG_EVERY = 20
TRAIN_SEED = 42

torch.manual_seed(TRAIN_SEED)

# Новый генератор с воспроизводимым sampling.
training_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=TRAIN_SEED,
)

# Совершенно новая модель.
profile_model = create_cnn_baseline().to(
    device
)

optimizer = DirectMLAdam(
    profile_model.parameters(),
    lr=1e-3,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.0,
)

profile_model.train()

loss_history = []
ema_loss = None
seen_position_strand = 0
start_time = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(1, TRAIN_STEPS + 1):
        train_batch = (
            training_generator.sample_batch(
                batch_size=TRAIN_BATCH_SIZE
            )
        )

        x_both_gpu = (
            train_batch
            .x_both_strands
            .to(device)
        )

        plus_target_gpu = (
            train_batch
            .plus_target
            .to(device)
        )

        minus_target_reverse_gpu = (
            train_batch
            .minus_target_reverse
            .to(device)
        )

        plus_weight_gpu = (
            train_batch
            .plus_loss_weight
            .to(device)
        )

        minus_weight_reverse_gpu = (
            train_batch
            .minus_loss_weight_reverse
            .to(device)
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_logits,
            minus_reverse_logits,
        ) = forward_both_strands(
            profile_model,
            x_both_gpu,
            batch_size=TRAIN_BATCH_SIZE,
            context_flank=(
                profile_config.context_flank
            ),
            target_length=(
                profile_config.target_length
            ),
        )

        loss = (
            dml_weighted_profile_bce_with_logits(
                plus_logits,
                minus_reverse_logits,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_weight_gpu,
                minus_weight_reverse_gpu,
            )
        )

        loss.backward()
        optimizer.step()

        loss_value = loss.item()
        loss_history.append(loss_value)

        if ema_loss is None:
            ema_loss = loss_value
        else:
            ema_loss = (
                0.95 * ema_loss
                + 0.05 * loss_value
            )

        seen_position_strand += int(
            train_batch.mask.sum()
        )

        if (
            step == 1
            or step % LOG_EVERY == 0
        ):
            elapsed = (
                time.perf_counter()
                - start_time
            )

            throughput = (
                seen_position_strand
                / elapsed
            )

            print(
                f"step={step:4d} "
                f"loss={loss_value:.6f} "
                f"ema={ema_loss:.6f} "
                f"positions/s={throughput:,.0f}"
            )

fallback_warnings = [
    str(warning.message)
    for warning in caught
    if "fall back" in str(warning.message).lower()
]

elapsed = time.perf_counter() - start_time

print()
print("Общее время:", round(elapsed, 2), "сек")
print(
    "Время одного шага:",
    round(elapsed / TRAIN_STEPS, 4),
    "сек",
)
print(
    "Увидено position-strand:",
    seen_position_strand,
)
print(
    "Последний EMA loss:",
    round(ema_loss, 6),
)
print(
    "CPU fallback:",
    fallback_warnings,
)

assert len(loss_history) == TRAIN_STEPS
assert not fallback_warnings

step=   1 loss=0.692867 ema=0.692867 positions/s=211,222
step=  20 loss=0.615659 ema=0.686539 positions/s=235,160
step=  40 loss=0.598136 ema=0.660771 positions/s=238,192
step=  60 loss=0.554443 ema=0.645109 positions/s=240,420
step=  80 loss=0.523847 ema=0.594415 positions/s=241,600
step= 100 loss=0.612398 ema=0.597262 positions/s=245,037
step= 120 loss=0.415079 ema=0.555183 positions/s=246,215
step= 140 loss=0.673016 ema=0.557661 positions/s=246,842
step= 160 loss=0.393871 ema=0.565655 positions/s=246,499
step= 180 loss=0.536972 ema=0.525859 positions/s=247,605
step= 200 loss=0.538748 ema=0.535018 positions/s=247,856

Общее время: 6.1 сек
Время одного шага: 0.0305 сек
Увидено position-strand: 1511530
Последний EMA loss: 0.535018
CPU fallback: []


In [32]:
import math
import time
import warnings
from pathlib import Path

BASELINE_SEED = 42
BASELINE_STEPS = 10_000
BASELINE_BATCH_SIZE = 4
LOG_EVERY = 250
CHECKPOINT_STEPS = {2_000, 5_000, 10_000}

BASELINE_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_baseline"
    / "baseline_v1"
)

BASELINE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def model_state_on_cpu(model):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor
        in model.state_dict().items()
    }


def tree_to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()

    if isinstance(value, dict):
        return {
            key: tree_to_cpu(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            tree_to_cpu(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            tree_to_cpu(item)
            for item in value
        )

    return value


torch.manual_seed(BASELINE_SEED)

baseline_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=BASELINE_SEED,
)

baseline_model = create_cnn_baseline().to(
    device
)

baseline_optimizer = DirectMLAdam(
    baseline_model.parameters(),
    lr=1e-3,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.0,
)

baseline_model.train()

baseline_history = []
ema_loss = None
seen_position_strand = 0
start_time = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(
        1,
        BASELINE_STEPS + 1,
    ):
        batch = baseline_generator.sample_batch(
            batch_size=BASELINE_BATCH_SIZE
        )

        x_both_gpu = (
            batch
            .x_both_strands
            .to(device)
        )

        plus_target_gpu = (
            batch
            .plus_target
            .to(device)
        )

        minus_target_reverse_gpu = (
            batch
            .minus_target_reverse
            .to(device)
        )

        plus_weight_gpu = (
            batch
            .plus_loss_weight
            .to(device)
        )

        minus_weight_reverse_gpu = (
            batch
            .minus_loss_weight_reverse
            .to(device)
        )

        baseline_optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_logits,
            minus_reverse_logits,
        ) = forward_both_strands(
            baseline_model,
            x_both_gpu,
            batch_size=BASELINE_BATCH_SIZE,
            context_flank=(
                profile_config.context_flank
            ),
            target_length=(
                profile_config.target_length
            ),
        )

        loss = (
            dml_weighted_profile_bce_with_logits(
                plus_logits,
                minus_reverse_logits,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_weight_gpu,
                minus_weight_reverse_gpu,
            )
        )

        loss_value = loss.item()

        if not math.isfinite(loss_value):
            raise FloatingPointError(
                f"Non-finite loss at step {step}: "
                f"{loss_value}"
            )

        loss.backward()
        baseline_optimizer.step()

        if ema_loss is None:
            ema_loss = loss_value
        else:
            ema_loss = (
                0.99 * ema_loss
                + 0.01 * loss_value
            )

        seen_position_strand += int(
            batch.mask.sum()
        )

        if (
            step == 1
            or step % LOG_EVERY == 0
        ):
            elapsed = (
                time.perf_counter()
                - start_time
            )

            record = {
                "step": step,
                "loss": loss_value,
                "ema_loss": ema_loss,
                "elapsed_seconds": elapsed,
                "positions_per_second": (
                    seen_position_strand
                    / elapsed
                ),
                "seen_position_strand": (
                    seen_position_strand
                ),
            }

            baseline_history.append(record)

            print(
                f"step={step:5d} "
                f"loss={loss_value:.6f} "
                f"ema={ema_loss:.6f} "
                f"positions/s="
                f"{record['positions_per_second']:,.0f}"
            )

        if step in CHECKPOINT_STEPS:
            checkpoint_path = (
                BASELINE_DIR
                / f"model_step_{step:05d}.pt"
            )

            torch.save(
                {
                    "step": step,
                    "model_state_dict": (
                        model_state_on_cpu(
                            baseline_model
                        )
                    ),
                    "profile_config": {
                        "target_length": (
                            profile_config
                            .target_length
                        ),
                        "context_flank": (
                            profile_config
                            .context_flank
                        ),
                        "positive_branch_fraction": (
                            profile_config
                            .positive_branch_fraction
                        ),
                    },
                    "training_config": {
                        "seed": BASELINE_SEED,
                        "steps": BASELINE_STEPS,
                        "batch_size": (
                            BASELINE_BATCH_SIZE
                        ),
                        "learning_rate": 1e-3,
                    },
                },
                checkpoint_path,
            )

fallback_warnings = [
    str(warning.message)
    for warning in caught
    if "fall back" in str(warning.message).lower()
]

total_elapsed = (
    time.perf_counter()
    - start_time
)

final_checkpoint = {
    "step": BASELINE_STEPS,
    "model_state_dict": (
        model_state_on_cpu(
            baseline_model
        )
    ),
    "optimizer_state_dict": (
        tree_to_cpu(
            baseline_optimizer.state_dict()
        )
    ),
    "generator_rng_state": (
        baseline_generator
        .rng
        .bit_generator
        .state
    ),
    "torch_rng_state": (
        torch.get_rng_state()
    ),
    "profile_config": {
        "target_length": (
            profile_config.target_length
        ),
        "context_flank": (
            profile_config.context_flank
        ),
        "positive_branch_fraction": (
            profile_config
            .positive_branch_fraction
        ),
    },
    "training_config": {
        "seed": BASELINE_SEED,
        "steps": BASELINE_STEPS,
        "batch_size": BASELINE_BATCH_SIZE,
        "learning_rate": 1e-3,
        "betas": (0.9, 0.999),
        "eps": 1e-8,
        "weight_decay": 0.0,
    },
    "history": baseline_history,
    "seen_position_strand": (
        seen_position_strand
    ),
    "elapsed_seconds": total_elapsed,
}

final_checkpoint_path = (
    BASELINE_DIR
    / "baseline_final.pt"
)

torch.save(
    final_checkpoint,
    final_checkpoint_path,
)

history_path = (
    BASELINE_DIR
    / "training_history.csv"
)

pd.DataFrame(
    baseline_history
).to_csv(
    history_path,
    index=False,
)

print()
print(
    "Общее время:",
    round(total_elapsed, 2),
    "сек",
)

print(
    "Время шага:",
    round(
        total_elapsed / BASELINE_STEPS,
        4,
    ),
    "сек",
)

print(
    "Увидено position-strand:",
    seen_position_strand,
)

print(
    "Финальный EMA loss:",
    round(ema_loss, 6),
)

print(
    "CPU fallback:",
    fallback_warnings,
)

print(
    "Checkpoint:",
    final_checkpoint_path,
)

print(
    "История:",
    history_path,
)

assert not fallback_warnings
assert final_checkpoint_path.is_file()
assert history_path.is_file()

step=    1 loss=0.692867 ema=0.692867 positions/s=208,079
step=  250 loss=0.499741 ema=0.569263 positions/s=239,255
step=  500 loss=0.512836 ema=0.510874 positions/s=240,022
step=  750 loss=0.586017 ema=0.507534 positions/s=240,552
step= 1000 loss=0.318802 ema=0.496531 positions/s=240,638
step= 1250 loss=0.320959 ema=0.493530 positions/s=241,406
step= 1500 loss=0.427012 ema=0.476043 positions/s=241,584
step= 1750 loss=0.522727 ema=0.466288 positions/s=241,022
step= 2000 loss=0.320252 ema=0.465625 positions/s=240,330
step= 2250 loss=0.461387 ema=0.471095 positions/s=240,229
step= 2500 loss=0.513951 ema=0.464828 positions/s=239,645
step= 2750 loss=0.630885 ema=0.462734 positions/s=238,855
step= 3000 loss=0.599342 ema=0.459897 positions/s=237,969
step= 3250 loss=0.718059 ema=0.449434 positions/s=237,459
step= 3500 loss=0.488151 ema=0.433690 positions/s=237,053
step= 3750 loss=0.356846 ema=0.453345 positions/s=236,653
step= 4000 loss=0.430253 ema=0.441629 positions/s=236,755
step= 4250 los

In [ ]:
from ml_project.epic_cnn.evaluation import (
    evaluate_profile_model,
    load_cnn_baseline_checkpoint,
)

CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts/experiments/CNN_baseline/baseline_v1/baseline_final.pt"
)

loaded = load_cnn_baseline_checkpoint(
    CHECKPOINT_PATH,
    device=device,
)
validation_model = loaded.model
validation_config = loaded.profile_config
checkpoint = loaded.checkpoint

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=validation_config,
    seed=42,
)

print("Checkpoint step:", checkpoint["step"])
print("Validation tiles:", len(validation_generator.tile_index))
print("Validation position-strand:", int(validation_generator.tile_index["allowed_position_strand"].sum()))
print("Validation positives:", validation_generator.total_positive)
print("Device:", next(validation_model.parameters()).device)

In [ ]:
validation_result = evaluate_profile_model(
    validation_model,
    validation_generator,
    device=device,
    batch_size=8,
    score_decimals=5,
    progress_every_batches=500,
)

In [ ]:
VALIDATION_METRICS_PATH = CHECKPOINT_PATH.parent / "validation_metrics.csv"
validation_result.metrics.to_csv(VALIDATION_METRICS_PATH, index=False)

display(validation_result.metrics)

print(f"Validation time: {validation_result.elapsed_seconds:.2f} sec")
print(f"Position-strand/s: {validation_result.positions_per_second:,.0f}")
print("CPU fallback:", list(validation_result.fallback_warnings))
print("Metrics:", VALIDATION_METRICS_PATH)